# Notebook 01 — Data Exploration & Cleaning
## CIC-IDS-2017 · Network Attack Detection

This notebook covers the full ingestion, exploration, and cleaning pipeline for
the **CIC-IDS-2017** dataset and produces the train/test splits that are
**shared by both Task 1 (binary) and Task 2 (multiclass)**.

> *Résumé (FR) : Ce notebook charge les 8 CSV du CIC-IDS-2017 en float32 (mémoire),
> effectue une EDA complète, nettoie les données (infinis, colonnes constantes,
> doublons), encode les labels et réalise le découpage train/test stratifié sur
> le label multiclasse. Les indices du split sont sauvegardés pour être réutilisés
> par les notebooks 02 et 03.*

---
### Pipeline overview
```
8 CSV files  (latin-1 encoding, float32 downcast per file)
  ↓ load_and_merge_datasets()   merge + label distribution check (B2)
  ↓ clean_data()                inf→NaN, constant cols, NaN rows, dedup (B3/B6)
  ↓ encode_labels()             → (X, y_multi, y_bin)  y_bin from string (B1)
  ↓ make_split()                80/20 stratified on y_multi, save indices (B4)
  ↓ drop_near_zero_cols()       computed on X_TRAIN only (B6)
  ↓ scale_features()            fit on train, save scaler (B5)
  ↓ cache/splits.npz            reused by notebooks 02 & 03
```


## 1. Setup
> *Importation des bibliothèques et configuration.*

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath('..'))
from src import preprocessing as P
from src import config as C

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

C.ensure_dirs()
os.makedirs('../results/figures', exist_ok=True)
os.makedirs('../cache', exist_ok=True)

FIGURES    = '../results/figures'
CACHE_DIR  = '../cache'
DATASET    = '../dataset/'
SEED       = 42
print("Setup complete.")


## 2. Loading the Dataset

The CIC-IDS-2017 dataset is split across **8 CSV files** (Mon–Fri).  
Two important design choices here:

**Memory (8 GB RAM):** Each file is immediately downcast from float64 to float32
after reading and before concatenation, halving per-row memory from ~680 to ~340 bytes.
Only one file at a time is held in full float64 during this loop.

**Encoding (B2-fix):** `encoding='latin-1'` fixes an encoding issue in the
web-attack labels of the Thursday WebAttacks CSV.  The `norm()` function in
`encode_labels()` strips non-alphabetic characters during label normalisation.

After loading, the **raw label distribution** is printed to confirm all 15 classes
loaded correctly and no crash occurred on the WebAttacks file.

> *Chargement en float32 par fichier (mémoire). Encodage latin-1 pour le fichier
> WebAttacks (B2). Distribution brute des labels affichée pour vérification.*


In [ ]:
print("Loading and merging 8 CSV files ...")
df_raw = P.load_and_merge_datasets(DATASET)
print(f"Raw dataset: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")


## 3. First Look at the Data
> *Aperçu des types de données et premières lignes.*


In [ ]:
print("Column dtypes summary:")
print(df_raw.dtypes.value_counts())
print(f"\nMemory usage: {df_raw.memory_usage(deep=True).sum()/1e9:.2f} GB")
df_raw.head(3)


## 4. Label Distribution — Raw Data

The dataset is **heavily imbalanced**:
- BENIGN: ~80% of all flows
- Some attack classes have fewer than 50 samples (Heartbleed, Infiltration,
  Web Attack – SQL Injection)

This imbalance drives key design decisions for both tasks:
- Task 1 (binary): `class_weight='balanced'` or resampling
- Task 2 (multiclass): rare-class handling strategy (asked before training)

> *Distribution très déséquilibrée. BENIGN domine (~80%). Classes rares critiques
> pour la classification multiclasse.*


In [ ]:
label_counts_raw = df_raw['Label'].value_counts().sort_values(ascending=False)
print("Raw label distribution:")
print(label_counts_raw.to_string())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Left: all classes (log scale to show rare ones)
colors = ['#2196F3' if lbl == 'BENIGN' else '#F44336'
          for lbl in label_counts_raw.index]
label_counts_raw.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('Class Distribution — Raw (log scale)', fontsize=13)
axes[0].set_ylabel('Sample count')
axes[0].set_yscale('log')
axes[0].tick_params(axis='x', rotation=45)

# Right: binary view
binary_view = label_counts_raw.copy()
binary_view.index = ['BENIGN' if x == 'BENIGN' else 'ATTACK'
                     for x in binary_view.index]
binary_view = binary_view.groupby(binary_view.index).sum().sort_values(ascending=False)
binary_view.plot(kind='bar', ax=axes[1],
                 color=['#2196F3', '#F44336'], edgecolor='white')
axes[1].set_title('Binary View: BENIGN vs. ATTACK', fontsize=13)
axes[1].set_ylabel('Sample count')
axes[1].tick_params(axis='x', rotation=0)
total = binary_view.sum()
for p in axes[1].patches:
    pct = p.get_height() / total * 100
    axes[1].annotate(f'{int(p.get_height()):,}\n({pct:.1f}%)',
                     (p.get_x() + p.get_width()/2, p.get_height()/2),
                     ha='center', va='center', fontsize=12,
                     color='white', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{FIGURES}/01_class_distribution_raw.png', dpi=150, bbox_inches='tight')
plt.show()


## 5. Missing Values & Infinity Analysis (Before Cleaning)

CIC-IDS-2017 flow features include `Flow Bytes/s` and `Flow Packets/s`,
computed as `bytes ÷ duration`. When a flow has zero duration, this produces
`+∞` or `-∞`. Pandas reads these as IEEE-754 infinity — `df.isnull()` returns
**False** for them, so we must replace them with NaN explicitly before `dropna()`.

> *Les infinis (division par zéro dans les débits) ne sont pas détectés par
> `isnull()`. Remplacement explicite requis avant `dropna()`.*


In [ ]:
inf_mask  = df_raw.isin([float('inf'), float('-inf')])
nan_mask  = df_raw.isnull()
inf_cols  = inf_mask.sum()
nan_cols  = nan_mask.sum()

print("Columns with +/-inf:")
print(inf_cols[inf_cols > 0].to_string() or "  none")

print("\nColumns with NaN (before inf→NaN replacement):")
print(nan_cols[nan_cols > 0].to_string() or "  none")

prob_rows = (inf_mask | nan_mask).any(axis=1).sum()
print(f"\nRows with at least one inf or NaN: {prob_rows:,} "
      f"({prob_rows/len(df_raw)*100:.2f}%)")


## 6. Data Cleaning

We apply the shared `clean_data()` pipeline. Each step is explained below.

| Step | Why |
|---|---|
| `replace(inf, NaN)` | pandas doesn't detect inf with `isnull()` |
| Drop constant cols (`nunique≤1`) | zero information for any classifier (B6-fix: catches non-zero constants too) |
| `dropna()` | sklearn estimators cannot handle NaN or inf |
| `drop_duplicates()` | identical feature rows + label should only count once (B3-fix) |

**B6 note:** Near-zero column filtering (≥99% zeros) is **not done here**.
It is applied after the split, computed on X_TRAIN only, so we do not
use test-set statistics to decide which columns to keep.

**B3 extra:** The pipeline also counts feature-identical rows with different
labels (hash-based, vectorized) — these are rows with identical feature vectors but different labels —
the classifier cannot distinguish them.

> *Nettoyage : infinis, colonnes constantes, NaN, doublons. Le filtrage des
> colonnes quasi-nulles est reporté après le split (sur X_TRAIN uniquement, B6).*


In [ ]:
df_clean = P.clean_data(df_raw)
# df_raw no longer needed — free memory
del df_raw
import gc; gc.collect()
print(f"\nMemory freed. Clean dataset: {df_clean.memory_usage(deep=True).sum()/1e9:.2f} GB")


In [ ]:
# Post-cleaning label distribution
label_counts_clean = df_clean['Label'].value_counts().sort_values(ascending=False)
print("Label distribution after cleaning:")
print(label_counts_clean.to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
label_counts_clean.plot(kind='bar', ax=ax, color='darkorange', edgecolor='white')
ax.set_title('Class Distribution — After Cleaning (log scale)', fontsize=13)
ax.set_ylabel('Sample count')
ax.set_yscale('log')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(f'{FIGURES}/02_class_distribution_clean.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Label Encoding

We derive **two** target vectors from the same `Label` column.

**B1-fix — `y_bin` from raw string label:**  
`y_bin = (df_clean['Label'].str.strip() != 'BENIGN').astype(int)`  
This is built *before* numeric encoding and does not depend on the integer
label map.  Even if `label_names` is reordered, `y_bin` stays correct.

| Target | Type | Rule |
|---|---|---|
| `y_bin`   | int32 | ATTACK=1, BENIGN=0 — from raw string |
| `y_multi` | int32 | class index 0–14 |

Both targets are derived from the same `df_clean` rows, so indexing with
`train_idx` or `test_idx` gives the correct slice for either task.

> *B1 : y_bin construit depuis le nom de label brut, pas depuis l'encodage
> numérique. Les deux targets partagent les mêmes indices.*


In [ ]:
X, y_multi, y_bin = P.encode_labels(df_clean)

# B1 sanity check: y_bin==0 must equal the number of 'BENIGN' rows
n_benign_str = (df_clean['Label'].str.strip() == 'BENIGN').sum()
n_benign_bin = (y_bin == 0).sum()
assert n_benign_str == n_benign_bin, "B1 mismatch!"
print(f"X shape:            {X.shape}  (dtype: {X.dtypes.iloc[0]})")
print(f"y_multi unique:     {sorted(y_multi.unique())}")
print(f"y_bin:  BENIGN=0 → {int(n_benign_bin):,}   ATTACK=1 → {int((y_bin==1).sum()):,}")
print(f"B1 check (y_bin==0 == BENIGN rows in string): PASS")


## 8. Feature Correlation Heatmap (Top 20 by Variance)

Computed on a 100k-row sample to stay within the 8 GB RAM budget.
Pairs with |r| > 0.95 are printed — these may be redundant features that
Random Forest feature importance (notebook 02) can confirm.

> *Corrélations de Pearson sur un échantillon de 100k lignes pour économiser la RAM.*


In [ ]:
sample_size = min(100_000, len(X))
X_samp = X.sample(n=sample_size, random_state=SEED)
top20  = X_samp.var().nlargest(20).index
corr   = X_samp[top20].corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', center=0,
            linewidths=0.3, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Pearson Correlation — Top 20 Features by Variance (100k sample)',
             fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIGURES}/03_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("Strongly correlated pairs (|r| > 0.95):")
found = False
for i in corr.columns:
    for j in corr.columns:
        if i < j and abs(corr.loc[i, j]) > 0.95:
            print(f"  {i}  <->  {j} :  r = {corr.loc[i,j]:.3f}")
            found = True
if not found:
    print("  None above threshold.")
del X_samp


## 9. Train / Test Split (B4-fix)

**One split, shared by both tasks.**

We stratify on **`y_multi`** (15 classes), not `y_bin` (2 classes), because:
- Stratifying on the finer multiclass target guarantees every attack class —
  even the rarest (Heartbleed: ~11 samples) — appears in both train and test.
- Stratifying on `y_bin` would only guarantee binary proportions, possibly
  leaving Heartbleed entirely in one split.

The row **indices** are saved to `cache/` as `train_idx.npy` and `test_idx.npy`.
Notebooks 02 and 03 load these indices to slice their respective targets,
ensuring a byte-for-byte identical split.

> *B4 : découpage 80/20 stratifié sur y_multi (15 classes), pas sur y_bin.
> Les indices sont sauvegardés — notebooks 02 et 03 rechargent exactement
> ce même découpage.*


In [ ]:
train_idx, test_idx = P.make_split(
    n_samples=len(X),
    y_multi=y_multi,
    test_size=0.2,
    random_state=42,
    cache_dir=CACHE_DIR,
)

X_train_raw = X.iloc[train_idx].copy()
X_test_raw  = X.iloc[test_idx].copy()
y_train_multi = y_multi.iloc[train_idx].values
y_test_multi  = y_multi.iloc[test_idx].values
y_train_bin   = y_bin.iloc[train_idx].values
y_test_bin    = y_bin.iloc[test_idx].values

print(f"X_train (raw): {X_train_raw.shape}")
print(f"X_test  (raw): {X_test_raw.shape}")
print(f"Binary — train attack rate: {y_train_bin.mean()*100:.2f}%")
print(f"Binary — test  attack rate: {y_test_bin.mean()*100:.2f}%")


In [ ]:
# Stratification verification chart
CLASS_NAMES = [
    'BENIGN','FTP-Patator','SSH-Patator','DoS slowloris','DoS Slowhttptest',
    'DoS Hulk','DoS GoldenEye','Heartbleed','Web Attack BF','Web Attack XSS',
    'Web Attack SQLi','Infiltration','Bot','DDoS','PortScan'
]
trn_pct = pd.Series(y_train_multi).value_counts(normalize=True).sort_index() * 100
tst_pct = pd.Series(y_test_multi).value_counts(normalize=True).sort_index() * 100

fig, ax = plt.subplots(figsize=(14, 5))
x = range(len(CLASS_NAMES))
ax.bar([i - 0.2 for i in x], [trn_pct.get(i,0) for i in x],
       0.4, label='Train', color='steelblue', alpha=0.85)
ax.bar([i + 0.2 for i in x], [tst_pct.get(i,0) for i in x],
       0.4, label='Test',  color='darkorange', alpha=0.85)
ax.set_xticks(list(x))
ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Proportion (%)')
ax.set_yscale('log')
ax.set_title('Stratification Check — Class Proportions Train vs Test (log scale)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES}/04_stratification_check.png', dpi=150, bbox_inches='tight')
plt.show()
print("Max |train% - test%| per class:")
diffs = {CLASS_NAMES[i]: abs(trn_pct.get(i,0)-tst_pct.get(i,0)) for i in range(15)}
for name, d in sorted(diffs.items(), key=lambda kv: -kv[1]):
    print(f"  {name:25s}  {d:.5f}%")


## 10. Drop Near-Zero Columns (B6-fix — computed on X_TRAIN only)

**Why train-only?**  
Computing the % of zeros on the full dataset (or the test set) would
constitute feature-selection data leakage: the decision of which columns
to keep would depend on test-set statistics.

We compute zero-percentages exclusively on `X_train`, then drop the same
columns from both `X_train` and `X_test`.

> *B6 : le filtrage des colonnes quasi-nulles est calculé sur X_TRAIN uniquement,
> puis appliqué à X_TEST (pas de data leakage).*


In [ ]:
X_train_f, X_test_f, dropped_cols = P.drop_near_zero_cols(
    X_train_raw, X_test_raw, threshold=99.0
)
print(f"Features after near-zero filter: {X_train_f.shape[1]}")
if dropped_cols:
    print("Dropped columns:", dropped_cols)
del X_train_raw, X_test_raw


## 11. Feature Scaling (B5-fix — save scaler)

`StandardScaler` is fitted on `X_train` only, then applied to `X_test`.

**B5-fix:** The fitted scaler and the ordered list of feature names are saved
to `cache/` as `scaler.joblib` and `feature_names.joblib`.  These files are
needed for the Django `ids_project` dashboard to scale new incoming flows
with the **exact same parameters** as training.

> *B5 : le scaler ajusté sur X_TRAIN et les noms de features sont sauvegardés
> pour la prédiction en production (dashboard Django).*


In [ ]:
X_train, X_test, scaler = P.scale_features(
    X_train_f, X_test_f, cache_dir=CACHE_DIR
)
del X_train_f, X_test_f
import gc; gc.collect()
print(f"X_train dtype: {X_train.dtypes.iloc[0]}  "
      f"X_test dtype: {X_test.dtypes.iloc[0]}")


## 12. Save Processed Data to Cache

Notebooks 02 and 03 load from `cache/splits.npz` and `cache/train_idx.npy`
rather than re-running the full pipeline.  This guarantees the identical split.

> *Sauvegarde des arrays train/test pour les notebooks suivants.*


In [ ]:
import joblib

splits_path = f'{CACHE_DIR}/splits.npz'
np.savez_compressed(
    splits_path,
    X_train=X_train.values,
    X_test=X_test.values,
    y_train_bin=y_train_bin,
    y_test_bin=y_test_bin,
    y_train_multi=y_train_multi,
    y_test_multi=y_test_multi,
)
print(f"Saved splits.npz  ({os.path.getsize(splits_path)/1e6:.1f} MB)")
print(f"Cache directory contents:")
for fn in sorted(os.listdir(CACHE_DIR)):
    size_mb = os.path.getsize(f'{CACHE_DIR}/{fn}') / 1e6
    print(f"  {fn:30s}  {size_mb:.1f} MB")


## 13. Summary & Key Takeaways

| Step | Result |
|---|---|
| Raw dataset | ~3.1M rows × 85 columns |
| After cleaning | see output above |
| Constant columns dropped | see Step 6 output |
| NaN rows dropped | see Step 6 output |
| Duplicate rows dropped | see Step 6 output |
| Near-zero columns (train only) | see Step 10 output |
| Final features | see X_train.shape |
| Train set | 80% of cleaned data |
| Test set | 20% of cleaned data |

### Key design decisions for the oral defence
1. **`encoding='latin-1'`** (B2) — only way to load all 8 files without crashing.
2. **float32 downcast per file** (memory) — halves RAM usage on 8 GB laptop.
3. **`y_bin` from raw string** (B1) — `Label != 'BENIGN'`, independent of encoding index.
4. **Stratify on `y_multi`** (B4) — finer stratification ensures rare classes appear in both splits.
5. **Shared split via saved indices** (B4) — `train_idx.npy` guarantees notebooks 02 & 03 use the same rows.
6. **Near-zero filter on X_TRAIN only** (B6) — no test-set leakage into feature selection.
7. **Scaler saved** (B5) — required for inference on new network flows.

---
> *Prochaine étape → Notebook 02 : Classification Binaire (BENIGN vs ATTACK)*
